# Distilling Step by Step

Video tutorial: https://www.youtube.com/watch?v=tpMOF1cT4Fc
Tutorial code: https://github.com/simranjeet97/LLM_Distillation

Underlying paper:

Hsieh Et al. 2023: Distilling Step-by-Step! Outperforming Larger Language Models with Less Training Data and Smaller Model Sizes
paper: https://aclanthology.org/2023.findings-acl.507.pdf
video: https://aclanthology.org/2023.findings-acl.507.mp4

```
bibtex:
@inproceedings{hsieh-etal-2023-distilling,
    title = "Distilling Step-by-Step! Outperforming Larger Language Models with Less Training Data and Smaller Model Sizes",
    author = "Hsieh, Cheng-Yu  and
      Li, Chun-Liang  and
      Yeh, Chih-kuan  and
      Nakhost, Hootan  and
      Fujii, Yasuhisa  and
      Ratner, Alex  and
      Krishna, Ranjay  and
      Lee, Chen-Yu  and
      Pfister, Tomas",
    editor = "Rogers, Anna  and
      Boyd-Graber, Jordan  and
      Okazaki, Naoaki",
    booktitle = "Findings of the Association for Computational Linguistics: ACL 2023",
    month = jul,
    year = "2023",
    address = "Toronto, Canada",
    publisher = "Association for Computational Linguistics",
    url = "https://aclanthology.org/2023.findings-acl.507/",
    doi = "10.18653/v1/2023.findings-acl.507",
    pages = "8003--8017",
    abstract = "Deploying large language models (LLMs) is challenging because they are memory inefficient and compute-intensive for practical applications. In reaction, researchers train smaller task-specific models by either finetuning with human labels or distilling using LLM-generated labels. However, finetuning and distillation require large amounts of training data to achieve comparable performance to LLMs. We introduce Distilling step-by-step, a new mechanism that (a) trains smaller models that outperform LLMs, and (b) achieves so by leveraging less training data needed by finetuning or distillation. Our method extracts LLM rationales as additional supervision for training small models within a multi-task framework. We present three findings across 4 NLP benchmarks: First, compared to both finetuning and distillation, our mechanism achieves better performance with much fewer labeled/unlabeled training examples. Second, compared to few-shot prompted LLMs, we achieve better performance using substantially smaller model sizes. Third, we reduce both the model size and the amount of data required to outperform LLMs; our finetuned 770M T5 model outperforms the few-shot prompted 540B PaLM model using only 80{\%} of available data on a benchmark, whereas standard finetuning the same T5 model struggles to match even by using 100{\%} of the dataset."
}
```

In [2]:
# install
# python -m ...
# pip install torch torchvision --index-url https://download.pytorch.org/whl/cpu
# pip install transformers[torch]
# pip install datasets

# install ollama with `curl -fsSL https://ollama.com/install.sh | sh`
# download models e.g. `ollama pull deepseek-r1:1.5b`
# list ollama models: `ollama list`


# additionally
# pip install ipywidgets

import torch
from transformers import AutoTokenizer, AutoModelForMaskedLM, Trainer, TrainingArguments
from datasets import load_dataset

## Loading Student Model

In [3]:
# load student model
from transformers import AutoModelForSequenceClassification

STUDENT_MODEL = "google-bert/bert-base-uncased"  # from https://huggingface.co/google-bert/bert-base-uncased
st_model = AutoModelForSequenceClassification.from_pretrained(STUDENT_MODEL)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: google-bert/bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [4]:
# load dataset
# OLD: dataset = load_dataset("sst2", split="train[:10]")
dataset = load_dataset("stanfordnlp/sst2", split="train[:10]")

In [5]:
dataset

Dataset({
    features: ['idx', 'sentence', 'label'],
    num_rows: 10
})

In [8]:
# load teacher model (has to be a thinking model)
from langchain_ollama import ChatOllama

# initialize the chat model
llm_engine = ChatOllama(
    model="deepseek-r1:1.5b",   # first, download the model: ´ollama pull deepseek-r1:1.5b` (1.1 GB), 
                                # see then `/usr/share/ollama/.ollama/models/manifests/registry.ollama.ai/library/deepseek-r1/1.5b` 
                                # or `ollama list`
    base_url="http://localhost:11434",  # with default ollama port
    temperature=0.3
)

In [11]:
def generate_rationale(input_text):
    """
    Uses Ollama's DeepSeek R1 model to generate a step-by-step rationale for the given input.
    """
    prompt = f"Explain step-by-step reasoning before answering: {input_text}"
    response = llm_engine.invoke(prompt)  # Using LangChain's invoke method
    return response.content if hasattr(response, "content") else response

print(generate_rationale("Explain AI in one sentence."))

m = """
ISSUE: ConnectError: [Errno 111] Connection refused

Connection refused indicates the service is not exposed/listening on this address/port.
Is ollama configured to listen on 0.0.0.0? It only listens on localhost by default so if you want to use it remotely, configuring OLLAMA_HOST is a requirement

-> install ollama first
"""



Artificial Intelligence (AI) is a field of study that enables machines to perform tasks that typically require human intelligence.


In [14]:
# MISSING in tutorial:
# https://huggingface.co/docs/transformers/v5.12.0/en/model_doc/auto#transformers.AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(STUDENT_MODEL)
tokenizer

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

BertTokenizer(name_or_path='google-bert/bert-base-uncased', vocab_size=30522, model_max_length=512, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	100: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	101: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	102: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	103: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
})

In [15]:
# prepare dataset with rationales
def process_data(example: dict) -> dict:
    input_text = example["sentence"]  # change this depending on your dataset format
    rationale = generate_rationale(input_text)
    label = example["label"]

    # tokenize input and rationale
    input_enc = tokenizer(input_text, truncation=True, padding="max_length", max_length=256)
    rationale_enc = tokenizer(rationale, truncation=True, padding="max_length", max_length=256)

    return {
        "input_ids": input_enc["input_ids"],
        "attention_mask": input_enc["attention_mask"],
        "labels": label,
        "rationale_ids": rationale_enc["input_ids"],
        "rationalemask": rationale_enc["attention_mask"],
    }

# apply function to dataset
processed_dataset = dataset.map(process_data)

m = """Parameter 'function'=<function process_data at 0x74d470ba0b80> of the transform datasets.arrow_dataset.Dataset._map_single couldn't be hashed properly, 
a random hash was used instead. Make sure your transforms and parameters are serializable with pickle or dill for the dataset fingerprinting and caching to work. 
If you reuse this transform, the caching mechanism will consider it to be different from the previous calls and recompute everything. 
This warning is only shown once. Subsequent hashing failures won't be shown.
"""

Map:   0%|          | 0/10 [00:00<?, ? examples/s]

In [16]:
from datasets import Dataset

# assuming 'dataset' is your Dataset object
processed_dataset.save_to_disk("preprocessed_dataset")

Saving the dataset (0/1 shards):   0%|          | 0/10 [00:00<?, ? examples/s]

In [17]:
from datasets import load_from_disk

# load the dataset from the saved directory
processed_dataset = load_from_disk("preprocessed_dataset")
print(processed_dataset)

Dataset({
    features: ['idx', 'sentence', 'label', 'input_ids', 'attention_mask', 'labels', 'rationale_ids', 'rationalemask'],
    num_rows: 10
})


In [18]:
training_args = TrainingArguments(
    output_dir="./results",  # directory to save the model and checkpoints
    eval_strategy="epoch",
    learning_rate=5e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    save_strategy="epoch",
    push_to_hub=False,
)

In [21]:
class MultiTaskTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None, **kwargs):
        labels = inputs.pop("labels")
        rationale_ids = inputs.pop("rationale_ids", None)

        outputs = st_model(**inputs)
        
        loss_fn = torch.nn.CorssEntropyLoss()
        label_loss = loss_fn(outputs.logits, labels)

        if rationale_ids is not None:
            rationale_outputs = model(input_ids=rationale_ids, attention_mask=inputs["attention_mask"])
            rationale_loss = loss_fn(rationale_outputs.logits, rationale_ids)
            loss = label_loss + 0.5 * rationale_loss  # weighted loss
        else:
            loss = label_loss

        return (loss, outputs) if return_outputs else loss

trainer = MultiTaskTrainer(
    model=st_model,
    args=training_args,
    train_dataset=processed_dataset,
    eval_dataset=processed_dataset,
)

In [22]:
processed_dataset

Dataset({
    features: ['idx', 'sentence', 'label', 'input_ids', 'attention_mask', 'labels', 'rationale_ids', 'rationalemask'],
    num_rows: 10
})

In [ ]:
trainer.train()
trainer.save_model("./results")
print("Distillation complete. Smaller 'student' model saved.")

m = """/home/emm/anaconda3/envs/venv_ml-exp/lib/python3.13/site-packages/torch/utils/data/dataloader.py:752: 
UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
  """

/home/emm/anaconda3/envs/venv_ml-exp/lib/python3.13/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


## For AutoTrain

UI in huggingface, here, we do the same:

In [ ]:
import pandas as pd

# prepare data for the DataFrame
data = {
    "text": [],
    "rationale": [],
    "target": []
}

for example in dataset:
    input_text = example["sentence"]
    label = example["label"]
    rationale = generate_rationale(input_text)

    data["text"].append(input_text)
    data["rationale"].append(rationale)
    data["target"].append(label)


# create DataFrame
df = pd.DataFrame(data)

# save to csv
df.to_csv("train.csv", index=False)
